In [ ]:
!pip install torch torchvision segmentation-models-pytorch albumentations opencv-python matplotlib scikit-learn tqdm

In [6]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp

In [7]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

Using device: mps


In [8]:
BASE_DIR = "output_dataset"

image_dir = os.path.join(BASE_DIR, "images")
crop_mask_dir = os.path.join(BASE_DIR, "crop_mask")
damage_mask_dir = os.path.join(BASE_DIR, "damage_mask")

valid_ext = (".png", ".jpg", ".jpeg")

image_files = sorted([
    f for f in os.listdir(image_dir)
    if f.lower().endswith(valid_ext)
])

print("Total images:", len(image_files))

Total images: 541


In [9]:
train_files, temp_files = train_test_split(image_files, test_size=0.3, random_state=42)
val_files, test_files = train_test_split(temp_files, test_size=0.5, random_state=42)

print(len(train_files), len(val_files), len(test_files))

378 81 82


In [10]:
class CropDataset(Dataset):
    def __init__(self, file_list, transform=None):
        self.files = file_list
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        f = self.files[idx]

        image = cv2.cvtColor(cv2.imread(os.path.join(image_dir, f)), cv2.COLOR_BGR2RGB)

        crop_mask = cv2.imread(os.path.join(crop_mask_dir, f), 0)
        damage_mask = cv2.imread(os.path.join(damage_mask_dir, f), 0)

        crop_mask = (crop_mask > 127).astype(np.float32)
        damage_mask = (damage_mask > 127).astype(np.float32)

        damage_mask = damage_mask * crop_mask

        if self.transform:
            augmented = self.transform(image=image, masks=[crop_mask, damage_mask])
            image = augmented["image"]
            crop_mask, damage_mask = augmented["masks"]

        crop_mask = crop_mask.unsqueeze(0)
        damage_mask = damage_mask.unsqueeze(0)

        return image, crop_mask, damage_mask

In [35]:
train_transform = A.Compose([
    A.Resize(256, 256),

    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),

    A.RandomRotate90(p=0.5),

    A.Affine(
        translate_percent=0.05,
        scale=(0.95, 1.05),
        rotate=(-10, 10),
        p=0.5
    ),

    A.RandomBrightnessContrast(p=0.2),

    A.Normalize(mean=(0.485,0.456,0.406),
                std=(0.229,0.224,0.225)),
    ToTensorV2()
])

In [36]:
train_dataset = CropDataset(train_files, train_transform)
val_dataset = CropDataset(val_files, val_transform)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, num_workers=0)

In [37]:
model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights="imagenet",
    decoder_attention_type="scse",
    in_channels=3,
    classes=1
).to(device)

In [38]:
def dice_loss(pred, target):
    pred = torch.sigmoid(pred)
    intersection = (pred * target).sum()
    return 1 - (2 * intersection + 1e-6) / (pred.sum() + target.sum() + 1e-6)


def focal_loss(pred, target, alpha=0.8, gamma=2):
    pred = torch.sigmoid(pred)
    bce = -(target * torch.log(pred + 1e-6) + (1 - target) * torch.log(1 - pred + 1e-6))
    return (alpha * (1 - pred) ** gamma * bce).mean()


def loss_fn(pred, target):
    return dice_weight * dice_loss(pred, target) + \
           (1 - dice_weight) * focal_loss(pred, target)

In [39]:
def iou_score(pred, target):
    pred = (torch.sigmoid(pred) > 0.5).float()
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    return (intersection + 1e-6) / (union + 1e-6)

In [40]:
optimizer = torch.optim.Adam(model.parameters(), lr = 8.8e-05, weight_decay = 1.15e-06 )

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=3, factor=0.5
)

In [41]:
def dice_score(pred, target):
    pred = (torch.sigmoid(pred) > 0.5).float()
    intersection = (pred * target).sum()
    return (2 * intersection + 1e-6) / (pred.sum() + target.sum() + 1e-6)


def iou_score(pred, target):
    pred = (torch.sigmoid(pred) > 0.5).float()
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection
    return (intersection + 1e-6) / (union + 1e-6)


def mae_score(pred, target, crop_mask):
    pred = (torch.sigmoid(pred) > 0.5).float()
    error = torch.abs(pred - target) * crop_mask
    return error.mean()

In [42]:
num_epochs = 25
best_iou = 0

for epoch in range(num_epochs):
    print(f"\n================ Epoch {epoch+1}/{num_epochs} ================")

    # ---------------- TRAIN ----------------
    model.train()

    train_loss = 0
    train_iou = 0
    train_dice = 0
    train_mae = 0

    for i, (images, crop_masks, masks) in enumerate(train_loader):
        images = images.to(device)
        masks = masks.to(device)
        crop_masks = crop_masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = loss_fn(outputs, masks)
        loss.backward()
        optimizer.step()

        # Metrics
        train_loss += loss.item()
        train_iou += iou_score(outputs, masks).item()
        train_dice += dice_score(outputs, masks).item()
        train_mae += mae_score(outputs, masks, crop_masks).item()

        # 🔄 Progress print (every 20 batches)
        if i % 20 == 0:
            print(f"[Train] Batch {i}/{len(train_loader)} | Loss: {loss.item():.4f}")

    # Average train metrics
    train_loss /= len(train_loader)
    train_iou /= len(train_loader)
    train_dice /= len(train_loader)
    train_mae /= len(train_loader)

    # ---------------- VALIDATION ----------------
    model.eval()

    val_loss = 0
    val_iou = 0
    val_dice = 0
    val_mae = 0

    with torch.no_grad():
        for images, crop_masks, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)
            crop_masks = crop_masks.to(device)

            outputs = model(images)

            loss = loss_fn(outputs, masks)

            val_loss += loss.item()
            val_iou += iou_score(outputs, masks).item()
            val_dice += dice_score(outputs, masks).item()
            val_mae += mae_score(outputs, masks, crop_masks).item()

    # Average val metrics
    val_loss /= len(val_loader)
    val_iou /= len(val_loader)
    val_dice /= len(val_loader)
    val_mae /= len(val_loader)

    # ---------------- PRINT SUMMARY ----------------
    print("\n📊 TRAIN METRICS")
    print(f"Loss: {train_loss:.4f} | IoU: {train_iou:.4f} | Dice: {train_dice:.4f} | MAE: {train_mae:.4f}")

    print("📊 VALIDATION METRICS")
    print(f"Loss: {val_loss:.4f} | IoU: {val_iou:.4f} | Dice: {val_dice:.4f} | MAE: {val_mae:.4f}")

    # ---------------- LR SCHEDULER ----------------
    scheduler.step(val_iou)

    # ---------------- SAVE BEST ----------------
    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), "best_attunet_phase1.pth")
        print("✅ Best model saved!")


================ Epoch 1/25 ================
[Train] Batch 0/189 | Loss: 0.5820
[Train] Batch 20/189 | Loss: 0.5923
[Train] Batch 40/189 | Loss: 0.6990
[Train] Batch 60/189 | Loss: 0.4192
[Train] Batch 80/189 | Loss: 0.4436
[Train] Batch 100/189 | Loss: 0.5813
[Train] Batch 120/189 | Loss: 0.5751
[Train] Batch 140/189 | Loss: 0.3324
[Train] Batch 160/189 | Loss: 0.5655
[Train] Batch 180/189 | Loss: 0.2894

📊 TRAIN METRICS
Loss: 0.4836 | IoU: 0.2692 | Dice: 0.4104 | MAE: 0.2738
📊 VALIDATION METRICS
Loss: 0.4264 | IoU: 0.3476 | Dice: 0.5081 | MAE: 0.1666
✅ Best model saved!

================ Epoch 2/25 ================
[Train] Batch 0/189 | Loss: 0.3420
[Train] Batch 20/189 | Loss: 0.3314
[Train] Batch 40/189 | Loss: 0.4055
[Train] Batch 60/189 | Loss: 0.4415
[Train] Batch 80/189 | Loss: 0.3269
[Train] Batch 100/189 | Loss: 0.3145
[Train] Batch 120/189 | Loss: 0.5439
[Train] Batch 140/189 | Loss: 0.3799
[Train] Batch 160/189 | Loss: 0.4239
[Train] Batch 180/189 | Loss: 0.2711

📊 TRAIN M

# Optimization (Random Search)

In [28]:
def run_trial(lr, wd, encoder):

    model = smp.Unet(
        encoder_name=encoder,
        encoder_weights="imagenet",
        decoder_attention_type="scse",
        in_channels=3,
        classes=1
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)

    best_val_iou = 0

    # 🔥 SHORT TRAINING (IMPORTANT)
    for epoch in range(5):   # only 5 epochs

        model.train()
        for images, _, masks in train_loader:
            images, masks = images.to(device), masks.to(device)

            optimizer.zero_grad()
            outputs = model(images)

            loss = loss_fn(outputs, masks)
            loss.backward()
            optimizer.step()

        # Validation
        model.eval()
        val_iou = 0

        with torch.no_grad():
            for images, _, masks in val_loader:
                images, masks = images.to(device), masks.to(device)
                outputs = model(images)

                val_iou += iou_score(outputs, masks).item()

        val_iou /= len(val_loader)

        best_val_iou = max(best_val_iou, val_iou)

    return best_val_iou

In [29]:
import random

lrs = [5e-5, 1e-4, 2e-4]
wds = [0, 1e-5, 1e-4]
encoders = ["resnet34", "resnet50"]

best_config = None
best_score = 0

for trial in range(5):
    lr = random.choice(lrs)
    wd = random.choice(wds)
    enc = random.choice(encoders)

    print(f"\n🔍 Trial {trial+1}: lr={lr}, wd={wd}, encoder={enc}")

    score = run_trial(lr, wd, enc)

    print(f"Validation IoU: {score:.4f}")

    if score > best_score:
        best_score = score
        best_config = (lr, wd, enc)

print("\n🏆 BEST CONFIG:")
print(best_config, "IoU:", best_score)


🔍 Trial 1: lr=0.0002, wd=0, encoder=resnet34
Validation IoU: 0.4733

🔍 Trial 2: lr=0.0001, wd=0, encoder=resnet50
Validation IoU: 0.4847

🔍 Trial 3: lr=5e-05, wd=1e-05, encoder=resnet50
Validation IoU: 0.4769

🔍 Trial 4: lr=0.0001, wd=1e-05, encoder=resnet50
Validation IoU: 0.4961

🔍 Trial 5: lr=0.0001, wd=0, encoder=resnet50
Validation IoU: 0.4812

🏆 BEST CONFIG:
(0.0001, 1e-05, 'resnet50') IoU: 0.4961193036742327


# Optuna

In [43]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 11.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [optuna]━━━━ 4/5 [optuna]


In [44]:
import optuna

In [45]:
def objective(trial):

    # 🔥 Hyperparameters to tune
    lr = trial.suggest_loguniform("lr", 1e-5, 5e-4)
    wd = trial.suggest_loguniform("weight_decay", 1e-6, 1e-4)
    encoder = trial.suggest_categorical("encoder", ["resnet34", "resnet50"])
    dice_weight = trial.suggest_float("dice_weight", 0.5, 0.9)

    # ---------------- MODEL ----------------
    model = smp.Unet(
        encoder_name=encoder,
        encoder_weights="imagenet",
        decoder_attention_type="scse",
        in_channels=3,
        classes=1
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=wd
    )

    # ---------------- LOSS ----------------
    def tuned_loss(pred, target):
        return dice_weight * dice_loss(pred, target) + (1 - dice_weight) * focal_loss(pred, target)

    best_val_iou = 0

    # 🔥 SHORT TRAINING (VERY IMPORTANT)
    for epoch in range(5):  # keep small for speed

        # ---- TRAIN ----
        model.train()
        for images, _, masks in train_loader:
            images, masks = images.to(device), masks.to(device)

            optimizer.zero_grad()
            outputs = model(images)

            loss = tuned_loss(outputs, masks)
            loss.backward()
            optimizer.step()

        # ---- VALIDATION ----
        model.eval()
        val_iou = 0

        with torch.no_grad():
            for images, _, masks in val_loader:
                images, masks = images.to(device), masks.to(device)

                outputs = model(images)
                val_iou += iou_score(outputs, masks).item()

        val_iou /= len(val_loader)

        best_val_iou = max(best_val_iou, val_iou)

    return best_val_iou

In [46]:
study = optuna.create_study(direction="maximize")

study.optimize(objective, n_trials=10)   # 🔥 start with 10 trials

[I 2026-04-22 02:55:47,862] A new study created in memory with name: no-name-1ec0bac3-3c40-44ea-aed1-a74ccf08c5f6
/var/folders/cs/v4ktspys6y11p3px3st0_k100000gn/T/ipykernel_34294/3233492783.py:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("lr", 1e-5, 5e-4)
/var/folders/cs/v4ktspys6y11p3px3st0_k100000gn/T/ipykernel_34294/3233492783.py:5: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  wd = trial.suggest_loguniform("weight_decay", 1e-6, 1e-4)
[I 2026-04-22 03:03:36,495] Trial 0 finished with value: 0.46548467402051136 and parameters: {'lr': 9.67831535399795e-05, 'weight_decay': 9.903075052330226e-06, 'encoder': 'resnet34', 'dice_weight': 0.6

In [47]:
print("Best Trial:")
print(study.best_trial)

print("\nBest Params:")
print(study.best_params)

Best Trial:
FrozenTrial(number=3, state=<TrialState.COMPLETE: 1>, values=[0.48747055595967825], datetime_start=datetime.datetime(2026, 4, 22, 3, 18, 11, 939871), datetime_complete=datetime.datetime(2026, 4, 22, 3, 28, 14, 329754), params={'lr': 8.804154368532012e-05, 'weight_decay': 1.1542839113523317e-06, 'encoder': 'resnet50', 'dice_weight': 0.8023011241112356}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'lr': FloatDistribution(high=0.0005, log=True, low=1e-05, step=None), 'weight_decay': FloatDistribution(high=0.0001, log=True, low=1e-06, step=None), 'encoder': CategoricalDistribution(choices=('resnet34', 'resnet50')), 'dice_weight': FloatDistribution(high=0.9, log=False, low=0.5, step=None)}, trial_id=3, value=None)

Best Params:
{'lr': 8.804154368532012e-05, 'weight_decay': 1.1542839113523317e-06, 'encoder': 'resnet50', 'dice_weight': 0.8023011241112356}


In [49]:
best = study.best_params

model = smp.Unet(
    encoder_name=best["encoder"],
    encoder_weights="imagenet",
    decoder_attention_type="scse",
    in_channels=3,
    classes=1
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=best["lr"],
    weight_decay=best["weight_decay"]
)

def loss_fn(pred, target):
    return best["dice_weight"] * dice_loss(pred, target) + \
           (1 - best["dice_weight"]) * focal_loss(pred, target)

In [50]:
num_epochs = 25
best_iou = 0

for epoch in range(num_epochs):
    print(f"\n================ Epoch {epoch+1}/{num_epochs} ================")

    # ---------------- TRAIN ----------------
    model.train()

    train_loss = 0
    train_iou = 0
    train_dice = 0
    train_mae = 0

    for i, (images, crop_masks, masks) in enumerate(train_loader):
        images = images.to(device)
        masks = masks.to(device)
        crop_masks = crop_masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)

        loss = loss_fn(outputs, masks)
        loss.backward()
        optimizer.step()

        # Metrics
        train_loss += loss.item()
        train_iou += iou_score(outputs, masks).item()
        train_dice += dice_score(outputs, masks).item()
        train_mae += mae_score(outputs, masks, crop_masks).item()

        # 🔄 Progress print (every 20 batches)
        if i % 20 == 0:
            print(f"[Train] Batch {i}/{len(train_loader)} | Loss: {loss.item():.4f}")

    # Average train metrics
    train_loss /= len(train_loader)
    train_iou /= len(train_loader)
    train_dice /= len(train_loader)
    train_mae /= len(train_loader)

    # ---------------- VALIDATION ----------------
    model.eval()

    val_loss = 0
    val_iou = 0
    val_dice = 0
    val_mae = 0

    with torch.no_grad():
        for images, crop_masks, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)
            crop_masks = crop_masks.to(device)

            outputs = model(images)

            loss = loss_fn(outputs, masks)

            val_loss += loss.item()
            val_iou += iou_score(outputs, masks).item()
            val_dice += dice_score(outputs, masks).item()
            val_mae += mae_score(outputs, masks, crop_masks).item()

    # Average val metrics
    val_loss /= len(val_loader)
    val_iou /= len(val_loader)
    val_dice /= len(val_loader)
    val_mae /= len(val_loader)

    # ---------------- PRINT SUMMARY ----------------
    print("\n📊 TRAIN METRICS")
    print(f"Loss: {train_loss:.4f} | IoU: {train_iou:.4f} | Dice: {train_dice:.4f} | MAE: {train_mae:.4f}")

    print("📊 VALIDATION METRICS")
    print(f"Loss: {val_loss:.4f} | IoU: {val_iou:.4f} | Dice: {val_dice:.4f} | MAE: {val_mae:.4f}")

    # ---------------- LR SCHEDULER ----------------
    scheduler.step(val_iou)

    # ---------------- SAVE BEST ----------------
    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), "best_attunet_phase1.pth")
        print("✅ Best model saved!")


================ Epoch 1/25 ================
[Train] Batch 0/189 | Loss: 0.6188
[Train] Batch 20/189 | Loss: 0.7475
[Train] Batch 40/189 | Loss: 0.7086
[Train] Batch 60/189 | Loss: 0.7516
[Train] Batch 80/189 | Loss: 0.5690
[Train] Batch 100/189 | Loss: 0.4722
[Train] Batch 120/189 | Loss: 0.4130
[Train] Batch 140/189 | Loss: 0.4290
[Train] Batch 160/189 | Loss: 0.3871
[Train] Batch 180/189 | Loss: 0.6110

📊 TRAIN METRICS
Loss: 0.5549 | IoU: 0.2585 | Dice: 0.3963 | MAE: 0.2900
📊 VALIDATION METRICS
Loss: 0.4721 | IoU: 0.3776 | Dice: 0.5414 | MAE: 0.1464
✅ Best model saved!

================ Epoch 2/25 ================
[Train] Batch 0/189 | Loss: 0.3366
[Train] Batch 20/189 | Loss: 0.4583
[Train] Batch 40/189 | Loss: 0.4965
[Train] Batch 60/189 | Loss: 0.4237
[Train] Batch 80/189 | Loss: 0.4372
[Train] Batch 100/189 | Loss: 0.3205
[Train] Batch 120/189 | Loss: 0.3295
[Train] Batch 140/189 | Loss: 0.3946
[Train] Batch 160/189 | Loss: 0.3956
[Train] Batch 180/189 | Loss: 0.3968

📊 TRAIN M

# Final Optimization

In [11]:
def tta_predict(model, image, device):
    model.eval()
    
    image = image.unsqueeze(0).to(device)

    preds = []

    with torch.no_grad():
        # Original
        preds.append(torch.sigmoid(model(image)))

        # Horizontal flip
        img = torch.flip(image, dims=[3])
        pred = torch.sigmoid(model(img))
        preds.append(torch.flip(pred, dims=[3]))

        # Vertical flip
        img = torch.flip(image, dims=[2])
        pred = torch.sigmoid(model(img))
        preds.append(torch.flip(pred, dims=[2]))

        # HV flip
        img = torch.flip(image, dims=[2,3])
        pred = torch.sigmoid(model(img))
        preds.append(torch.flip(pred, dims=[2,3]))

    final_pred = torch.mean(torch.stack(preds), dim=0)

    return final_pred

In [12]:
import numpy as np

def find_best_threshold(model, val_loader, device):

    thresholds = np.arange(0.3, 0.7, 0.05)
    best_th = 0.5
    best_iou = 0

    for th in thresholds:
        iou_total = 0

        for images, _, masks in val_loader:
            images, masks = images.to(device), masks.to(device)

            preds = tta_predict(model, images[0], device)
            preds_bin = (preds > th).float()

            iou_total += iou_score(preds_bin, masks[0]).item()

        avg_iou = iou_total / len(val_loader)

        print(f"Threshold {th:.2f} → IoU: {avg_iou:.4f}")

        if avg_iou > best_iou:
            best_iou = avg_iou
            best_th = th

    print("\n🔥 Best Threshold:", best_th)
    return best_th

In [13]:
def evaluate_final(model, loader, device, threshold):

    model.eval()

    total_iou = 0
    total_dice = 0
    total_mae = 0

    with torch.no_grad():
        for images, crop_masks, masks in loader:
            images = images.to(device)
            masks = masks.to(device)
            crop_masks = crop_masks.to(device)

            preds = tta_predict(model, images[0], device)
            preds_bin = (preds > threshold).float()

            total_iou += iou_score(preds_bin, masks[0]).item()
            total_dice += dice_score(preds_bin, masks[0]).item()
            total_mae += mae_score(preds_bin, masks[0], crop_masks[0]).item()

    n = len(loader)

    print("\n🎯 FINAL RESULTS (Optimized)")
    print(f"IoU:  {total_iou/n:.4f}")
    print(f"Dice: {total_dice/n:.4f}")
    print(f"MAE:  {total_mae/n:.4f}")

In [14]:
best_th = find_best_threshold(model, val_loader, device)

evaluate_final(model, test_loader, device, best_th)

NameError: name 'model' is not defined

In [15]:
def ensemble_predict(models, image, device):
    preds = []

    for model in models:
        preds.append(tta_predict(model, image, device))

    return torch.mean(torch.stack(preds), dim=0)